# Macro Environment

In [3]:
import requests
import pandas as pd

In [14]:
url = "https://clinicaltrials.gov/api/v2/studies"

europe_countries = [
    "Germany", "France", "Switzerland", "United Kingdom", "Spain",
    "Italy", "Netherlands", "Belgium", "Sweden", "Denmark",
    "Austria", "Norway", "Finland", "Ireland", "Poland"
]

rows = []

for country in europe_countries:
    params = {
        "query.cond": "oncology OR genomics OR liquid biopsy",
        "query.locn": country,
        "filter.overallStatus": "RECRUITING,ACTIVE_NOT_RECRUITING,NOT_YET_RECRUITING",
        "pageSize": 100,
        "format": "json"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    for study in data.get("studies", []):
        protocol = study.get("protocolSection", {})
        identification = protocol.get("identificationModule", {})
        status = protocol.get("statusModule", {})
        design = protocol.get("designModule", {})
        sponsor = protocol.get("sponsorCollaboratorsModule", {})
        conditions = protocol.get("conditionsModule", {})
        dates = status.get("startDateStruct", {})

        rows.append({
            "country_search": country,
            "nct_id": identification.get("nctId"),
            "title": identification.get("briefTitle"),
            "status": status.get("overallStatus"),
            "start_date": dates.get("date"),
            "phase": design.get("phases", []),
            "sponsor": sponsor.get("leadSponsor", {}).get("name"),
            "conditions": conditions.get("conditions", [])
        })

df_trials = pd.DataFrame(rows)

print(df_trials.head())
    

  country_search       nct_id  \
0        Germany  NCT04925583   
1        Germany  NCT04772937   
2        Germany  NCT06821880   
3        Germany  NCT02474641   
4        Germany  NCT06522737   

                                               title                 status  \
0  Magnetic Resonance Guided Adaptive Stereotacti...  ACTIVE_NOT_RECRUITING   
1  Comparison of LLETZ Versus LEEP for the Treatm...             RECRUITING   
2  Developing a Patient-Reported Outcome (PRO) Sc...             RECRUITING   
3  Hypofractionation With Simultaneous Integrated...  ACTIVE_NOT_RECRUITING   
4  A Study of Duvelisib Versus Gemcitabine or Ben...             RECRUITING   

   start_date     phase                                 sponsor  \
0  2021-11-01  [PHASE1]          University Hospital Heidelberg   
1  2021-06-07      [NA]               Ruhr University of Bochum   
2  2025-02-03        []            Medical University Innsbruck   
3  2015-06-16      [NA]  University Hospital Schleswig-Hol

In [15]:
df_trials.columns

Index(['country_search', 'nct_id', 'title', 'status', 'start_date', 'phase',
       'sponsor', 'conditions'],
      dtype='object')

In [16]:
df_trials['country_search'].unique()

array(['Germany', 'France', 'Switzerland', 'United Kingdom', 'Spain',
       'Italy', 'Netherlands', 'Belgium', 'Sweden', 'Denmark', 'Austria',
       'Norway', 'Finland', 'Ireland', 'Poland'], dtype=object)

In [5]:
url2="https://api.reporter.nih.gov/v2/projects/search"

payload = {
    "criteria": {
        "advanced_text_search": {
            "operator": "and",
            "search_field": "all",
            "search_text": "genomics OR diagnostics OR oncology"
        },
        "fiscal_years": [2024, 2025]
    },
    "limit": 100,
    "offset": 0
}

response = requests.post(url2, json=payload)
data = response.json()
rows = []

for project in data["results"]:
    org = project.get("organization", {})
    rows.append({
        "project_title": project.get("project_title"),
        "fiscal_year": project.get("fiscal_year"),
        "award_amount": project.get("award_amount"),
        "institution": org.get("org_name"),
        "city": org.get("org_city"),
        "country": org.get("org_country"),
        "abstract": project.get("abstract_text")
    })

df_nih = pd.DataFrame(rows)

print(df_nih.head())


                                       project_title  fiscal_year  \
0                     Developmental Research Program         2025   
1  High-Throughput Genomics and Bioinformatic Ana...         2025   
2  Matching genotypes with personalized therapies...         2025   
3  Commercialization of a rapid, automated Hi-C p...         2025   
4  ECOG-ACRIN Integrated Leukemia Translational S...         2025   

   award_amount                                        institution  \
0       81247.0                UNIVERSITY OF MICHIGAN AT ANN ARBOR   
1      275705.0  UTAH STATE HIGHER EDUCATION SYSTEM--UNIVERSITY...   
2      407737.0                           JOHNS HOPKINS UNIVERSITY   
3      746680.0                                    CANTATA BIO LLC   
4      848075.0                  SLOAN-KETTERING INST CAN RESEARCH   

             city        country  \
0       ANN ARBOR  UNITED STATES   
1  SALT LAKE CITY  UNITED STATES   
2       BALTIMORE  UNITED STATES   
3       Worcester  U

In [8]:
df_nih.columns

Index(['project_title', 'fiscal_year', 'award_amount', 'institution', 'city',
       'country', 'abstract'],
      dtype='object')

In [10]:
df_nih['country'].unique()

array(['UNITED STATES'], dtype=object)

In [11]:
df_nih['city'].unique()

array(['ANN ARBOR', 'SALT LAKE CITY', 'BALTIMORE', 'Worcester',
       'NEW YORK', 'DURHAM', 'Columbus', 'NEW ORLEANS', 'STANFORD',
       'PORTLAND', 'SAINT LOUIS', 'OSWEGO', 'LEXINGTON', 'BOSTON',
       'Newark', 'PHILADELPHIA', None, 'ARCADIA', 'ATLANTA', 'SEATTLE',
       'Aurora', 'MEMPHIS', 'ALBUQUERQUE', 'WASHINGTON', 'San Diego',
       'LOS ANGELES', 'DALLAS', 'HOUSTON', 'MILWAUKEE', 'RICHLAND',
       'ROCHESTER', 'MADISON', 'Decatur', 'BURLINGTON', 'SAN FRANCISCO',
       'HONOLULU', 'BAR HARBOR', 'SAN DIEGO', 'MATHER', 'Pasadena',
       'OAKLAND'], dtype=object)

In [17]:
url = "https://data.europa.eu/api/hub/search/datasets/cordis-eu-research-projects-under-horizon-europe-2021-2027"

response = requests.get(url)

data = response.json()

print(data.keys())
print(data["result"].keys())

dict_keys(['result'])
dict_keys(['country', 'keywords', 'catalog', 'subject', 'translation_meta', 'description', 'language', 'title', 'accrual_periodicity', 'access_right', 'relation', 'spatial_resource', 'modified', 'id', 'categories', 'adms_identifier', 'issued', 'spatial', 'temporal', 'resource', 'landing_page', 'distributions', 'is_hvd', 'quality_meas', 'publisher', 'catalog_record', 'contact_point'])
